In [ ]:
"""
=============================================================================
  DEEP LEARNING OPTIMIZATION TECHNIQUES — Complete Guide with PyTorch
=============================================================================
  Covers: SGD, Momentum, Nesterov, Adagrad, Adadelta, RMSprop,
          Adam, AdamW, NAdam, RAdam, AdaMax, LAMB (via torch-optimizer)
  Dataset: MNIST (auto-downloaded via torchvision)
  Output:  Printed training logs + saved comparison plots
=============================================================================
"""

import os
import time
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ──────────────────────────────────────────────────────────────────────────────
# 0.  GLOBAL CONFIG
# ──────────────────────────────────────────────────────────────────────────────
EPOCHS      = 5
BATCH_SIZE  = 256
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR    = "./data"
RESULTS_DIR = "./optimizer_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 70)
print("  DEEP LEARNING OPTIMIZERS — Comprehensive Guide")
print("=" * 70)
print(f"  Device  : {DEVICE}")
print(f"  Epochs  : {EPOCHS}   |   Batch size: {BATCH_SIZE}")
print(f"  Dataset : MNIST (60 000 train / 10 000 test)")
print("=" * 70)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  DATASET  (MNIST — open-source, auto-download)
# ──────────────────────────────────────────────────────────────────────────────
print("\n[1] Loading MNIST dataset …")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"    Train samples : {len(train_dataset):,}")
print(f"    Test  samples : {len(test_dataset):,}")
print(f"    Input shape   : 28×28 grayscale → flattened to 784")


# ──────────────────────────────────────────────────────────────────────────────
# 2.  MODEL
# ──────────────────────────────────────────────────────────────────────────────
class MLP(nn.Module):
    """Simple 3-layer MLP for fair optimizer comparison."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

sample_model = MLP()
print(f"\n[2] Model: 3-layer MLP  |  Trainable params: {count_params(sample_model):,}")


# ──────────────────────────────────────────────────────────────────────────────
# 3.  OPTIMIZER DEFINITIONS + THEORY
# ──────────────────────────────────────────────────────────────────────────────
OPTIMIZER_CONFIGS = [

    # ── FIRST ORDER ──────────────────────────────────────────────────────────
    {
        "name"      : "SGD",
        "category"  : "First Order",
        "color"     : "#e74c3c",
        "theory"    : """
  STOCHASTIC GRADIENT DESCENT (SGD)
  ──────────────────────────────────
  The simplest optimizer. Computes gradient of loss w.r.t. each parameter
  and takes a small step in the negative gradient direction.

  Update rule:   θ ← θ − η · ∇L(θ; x_batch)

  Key hyperparameters:
    η  (lr)          : step size (typical: 0.01 – 0.1)

  Intuition: Imagine standing on a hilly landscape (loss surface) and always
  stepping downhill. With full-batch GD every step uses ALL data (expensive).
  SGD uses a random mini-batch, introducing noise that can actually help
  escape local minima — but convergence is slower and oscillatory.

  Best for: Final fine-tuning, when combined with a good LR schedule.
  Caution : Very sensitive to learning rate; needs hand-tuned scheduling.
        """,
        "build_optimizer": lambda params: torch.optim.SGD(params, lr=0.05)
    },

    {
        "name"      : "SGD + Momentum",
        "category"  : "First Order",
        "color"     : "#e67e22",
        "theory"    : """
  SGD WITH MOMENTUM
  ──────────────────
  Adds a velocity term that accumulates past gradients. Like a ball rolling
  downhill — it builds up speed in consistent directions and dampens the
  side-to-side oscillations you get in narrow loss valleys.

  Update rule:
    v  ← β · v  +  ∇L(θ)        (accumulate velocity)
    θ  ← θ − η · v               (update params)

  Key hyperparameters:
    η  (lr)       : step size   (typical: 0.01)
    β  (momentum) : decay rate  (typical: 0.9)

  Intuition: β = 0.9 means 90 % of last velocity is kept each step.
  This smooths the path towards the minimum and makes larger effective
  steps in persistent directions.

  Best for: Training CNNs on vision tasks; solid baseline.
        """,
        "build_optimizer": lambda params: torch.optim.SGD(params, lr=0.01, momentum=0.9)
    },

    {
        "name"      : "Nesterov",
        "category"  : "First Order",
        "color"     : "#f39c12",
        "theory"    : """
  NESTEROV ACCELERATED GRADIENT (NAG)
  ─────────────────────────────────────
  An improvement over standard Momentum: instead of computing the gradient
  at the CURRENT position θ, compute it at the LOOK-AHEAD position θ − β·v.
  This gives a more responsive correction — the optimizer "knows" where it's
  heading and adjusts before it gets there.

  Update rule:
    v  ← β · v  +  ∇L(θ − β·v)   (lookahead gradient)
    θ  ← θ − η · v

  Key hyperparameters:
    η  (lr)       : step size   (typical: 0.01)
    β  (momentum) : decay rate  (typical: 0.9)

  Intuition: Standard momentum is like a car accelerating then checking
  the map. Nesterov first checks the map (lookahead), then accelerates.
  Theoretical guarantee: O(1/k²) convergence vs O(1/k) for plain GD.

  Best for: Tasks where Momentum works but convergence speed matters.
        """,
        "build_optimizer": lambda params: torch.optim.SGD(
            params, lr=0.01, momentum=0.9, nesterov=True)
    },

    # ── ADAPTIVE ─────────────────────────────────────────────────────────────
    {
        "name"      : "Adagrad",
        "category"  : "Adaptive",
        "color"     : "#27ae60",
        "theory"    : """
  ADAGRAD (Adaptive Gradient Algorithm)
  ───────────────────────────────────────
  Adapts the learning rate PER PARAMETER by dividing by the square root of
  the cumulative sum of all past squared gradients.

  Update rule:
    G  ← G  +  g²              (accumulate squared gradients)
    θ  ← θ  −  (η / √(G + ε)) · g

  Key hyperparameters:
    η  (lr)  : global step size (typical: 0.01)
    ε        : numerical stability (1e-8)

  Intuition: Parameters that receive large/frequent gradients get smaller
  effective LR. Rare parameters (sparse embeddings in NLP) keep large LR.
  This is ideal for sparse data but has a FATAL FLAW: G only grows →
  effective LR → 0, so training eventually stalls.

  Best for: NLP with sparse gradients (word embeddings), early training.
  Caution : LR decay is monotonic — training stops improving eventually.
        """,
        "build_optimizer": lambda params: torch.optim.Adagrad(params, lr=0.01)
    },

    {
        "name"      : "Adadelta",
        "category"  : "Adaptive",
        "color"     : "#16a085",
        "theory"    : """
  ADADELTA
  ─────────
  Fixes Adagrad's vanishing LR by replacing the cumulative sum of past
  squared gradients with an EXPONENTIALLY DECAYING average (window).
  Remarkably, it also doesn't require setting a global learning rate at all.

  Update rule:
    E[g²]  ← ρ·E[g²]  +  (1−ρ)·g²          (running avg of squared grads)
    Δθ     = −(√(E[Δθ²] + ε) / √(E[g²] + ε)) · g
    E[Δθ²] ← ρ·E[Δθ²] +  (1−ρ)·Δθ²         (running avg of squared updates)
    θ      ← θ + Δθ

  Key hyperparameters:
    ρ   : smoothing constant (typical: 0.95)
    ε   : numerical stability (1e-6)

  Intuition: By using a ratio of recent gradient norms to recent update
  norms, the units naturally cancel and no manual LR is needed.

  Best for: Problems with rapidly changing gradient magnitudes.
        """,
        "build_optimizer": lambda params: torch.optim.Adadelta(params, lr=1.0, rho=0.95)
    },

    {
        "name"      : "RMSprop",
        "category"  : "Adaptive",
        "color"     : "#2980b9",
        "theory"    : """
  RMSPROP (Root Mean Square Propagation)
  ───────────────────────────────────────
  Proposed by Hinton (unpublished). Fixes Adagrad's diminishing LR by using
  an exponentially weighted moving average of squared gradients instead of
  their cumulative sum.

  Update rule:
    E[g²]  ← α·E[g²]  +  (1−α)·g²      (exponential moving avg)
    θ      ← θ − (η / √(E[g²] + ε)) · g

  Key hyperparameters:
    η  (lr)   : step size  (typical: 0.001)
    α  (alpha): decay rate (typical: 0.99)
    ε         : stability  (1e-8)

  Intuition: Like Adagrad but with a "fading memory" — only recent gradient
  history influences the adaptive scale. This prevents LR from collapsing.
  Originally designed for training RNNs (non-stationary gradients).

  Best for: RNNs, LSTMs, online learning, non-stationary objectives.
        """,
        "build_optimizer": lambda params: torch.optim.RMSprop(
            params, lr=0.001, alpha=0.99)
    },

    # ── ADAPTIVE MOMENT ───────────────────────────────────────────────────────
    {
        "name"      : "Adam",
        "category"  : "Adaptive Moment",
        "color"     : "#8e44ad",
        "theory"    : """
  ADAM (Adaptive Moment Estimation)
  ───────────────────────────────────
  The most widely used optimizer. Combines:
    • Momentum     : 1st moment (mean of gradients)
    • RMSprop      : 2nd moment (uncentered variance of gradients)
  with BIAS CORRECTION to account for zero-initialisation of moments.

  Update rule:
    m  ← β₁·m  +  (1−β₁)·g            (1st moment / "momentum")
    v  ← β₂·v  +  (1−β₂)·g²           (2nd moment / "velocity")
    m̂  = m / (1 − β₁ᵗ)                (bias-corrected 1st moment)
    v̂  = v / (1 − β₂ᵗ)                (bias-corrected 2nd moment)
    θ  ← θ − η · m̂ / (√v̂ + ε)

  Key hyperparameters:
    η  (lr)       : step size (typical: 1e-3)
    β₁            : 1st moment decay (typical: 0.9)
    β₂            : 2nd moment decay (typical: 0.999)
    ε             : stability (1e-8)

  Best for: Most tasks out of the box. The default choice for most DL work.
  Caution : Can converge to sharper minima than SGD; weight decay is broken.
        """,
        "build_optimizer": lambda params: torch.optim.Adam(
            params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8)
    },

    {
        "name"      : "AdamW",
        "category"  : "Adaptive Moment",
        "color"     : "#6c3483",
        "theory"    : """
  ADAMW (Adam with Decoupled Weight Decay)
  ──────────────────────────────────────────
  Fixes a subtle but important bug in Adam: when you add L2 regularisation
  (weight_decay) to Adam, the adaptive scaling interferes with the decay —
  parameters with small gradients are barely regularised. AdamW DECOUPLES
  weight decay from the gradient update.

  Update rule (key difference):
    Adam  → θ ← θ − η·m̂/(√v̂+ε) − η·λ·θ·(1/v̂)  ← WRONG coupling
    AdamW → θ ← θ − η·m̂/(√v̂+ε) − η·λ·θ          ← CORRECT decoupling

  Key hyperparameters:
    η            : step size    (typical: 1e-4)
    weight_decay : λ coefficient (typical: 0.01 – 0.1)

  Best for: Transformers, BERT, GPT, ViT — the standard optimizer for LLMs.
  Almost always preferred over Adam when weight decay is needed.
        """,
        "build_optimizer": lambda params: torch.optim.AdamW(
            params, lr=1e-3, betas=(0.9, 0.999), weight_decay=0.01)
    },

    {
        "name"      : "NAdam",
        "category"  : "Adaptive Moment",
        "color"     : "#c0392b",
        "theory"    : """
  NADAM (Nesterov-accelerated Adaptive Moment Estimation)
  ─────────────────────────────────────────────────────────
  Incorporates Nesterov's lookahead into Adam. Instead of applying the
  standard momentum term m̂ in the update, it uses the "next" momentum
  estimate — a one-step lookahead.

  Update rule (key change vs Adam):
    Standard Adam:  θ ← θ − η · m̂_t / (√v̂_t + ε)
    NAdam:          θ ← θ − η · [β₁·m̂_t + (1−β₁)·g_t/(1−β₁ᵗ)] / (√v̂_t + ε)

  Key hyperparameters: same as Adam + momentum_decay (≈ 4e-3)

  Intuition: By using the "lookahead" momentum, NAdam makes slightly more
  informed update steps. Small but consistent improvement over Adam on
  many tasks; particularly good for RNNs and fine-tuning.

  Best for: Tasks where Adam already works well, needing a small edge.
        """,
        "build_optimizer": lambda params: torch.optim.NAdam(
            params, lr=1e-3, betas=(0.9, 0.999))
    },

    {
        "name"      : "RAdam",
        "category"  : "Adaptive Moment",
        "color"     : "#d35400",
        "theory"    : """
  RADAM (Rectified Adam)
  ───────────────────────
  Early in Adam training, the adaptive LR has very HIGH VARIANCE because
  the second moment estimate is based on very few samples. This causes
  instability unless you manually add a LR warmup schedule. RAdam fixes
  this by AUTOMATICALLY switching between SGD (when variance is too high)
  and Adam (when variance is acceptable), based on the estimated variance
  of the adaptive LR.

  Update rule:
    Compute ρ_t = ρ∞ − 2t·β₂ᵗ / (1−β₂ᵗ)    (effective SMA length)
    IF ρ_t > 4:  use variance-rectified Adam update
    ELSE:        use plain SGD step (too noisy for adaptive)

  Intuition: Provides a principled, automatic warmup — no scheduler needed.
  In practice, RAdam ≈ Adam + warmup scheduler, but simpler.

  Best for: When you don't want to tune a warmup scheduler manually.
        """,
        "build_optimizer": lambda params: torch.optim.RAdam(
            params, lr=1e-3, betas=(0.9, 0.999))
    },

    {
        "name"      : "AdaMax",
        "category"  : "Adaptive Moment",
        "color"     : "#1abc9c",
        "theory"    : """
  ADAMAX (Adam generalised to L∞ norm)
  ──────────────────────────────────────
  A variant of Adam that replaces the L2 norm (squared gradient) in the
  second moment with the L∞ norm (maximum absolute gradient). The update
  step becomes controlled by the maximum recent gradient magnitude.

  Update rule:
    m  ← β₁·m + (1−β₁)·g          (1st moment, same as Adam)
    u  ← max(β₂·u, |g|)            (L∞ norm — no bias correction needed)
    θ  ← θ − (η/(1−β₁ᵗ)) · m/u

  Key insight: Since max() is naturally bounded, no bias-correction on u.
  This makes AdaMax more STABLE when gradient magnitudes vary greatly
  across parameters (e.g., large embedding tables with rare vs. frequent
  word updates).

  Best for: NLP models with embedding layers; large, sparse gradients.
        """,
        "build_optimizer": lambda params: torch.optim.Adamax(
            params, lr=2e-3, betas=(0.9, 0.999))
    },
]


# ──────────────────────────────────────────────────────────────────────────────
# 4.  TRAINING LOOP
# ──────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, optimizer, loader, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = F.cross_entropy(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += X.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out  = model(X)
            loss = F.cross_entropy(out, y)
            total_loss += loss.item() * X.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            total      += X.size(0)
    return total_loss / total, correct / total


# ──────────────────────────────────────────────────────────────────────────────
# 5.  RUN ALL OPTIMIZERS
# ──────────────────────────────────────────────────────────────────────────────
results = {}   # name → {"train_loss", "val_loss", "train_acc", "val_acc", "time"}

print("\n" + "=" * 70)
print("  TRAINING — running each optimizer for", EPOCHS, "epochs")
print("=" * 70)

for cfg in OPTIMIZER_CONFIGS:
    name = cfg["name"]
    # Print theory block
    print(f"\n{'─'*70}")
    print(cfg["theory"].strip())
    print(f"{'─'*70}")
    print(f"  ▶ Training with {name} …\n")

    # Fresh model + optimizer
    model     = MLP().to(DEVICE)
    optimizer = cfg["build_optimizer"](model.parameters())

    train_losses, val_losses  = [], []
    train_accs,   val_accs    = [], []
    t0 = time.time()

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, optimizer, train_loader, DEVICE)
        vl_loss, vl_acc = evaluate(model, test_loader, DEVICE)
        train_losses.append(tr_loss); val_losses.append(vl_loss)
        train_accs.append(tr_acc);   val_accs.append(vl_acc)
        print(f"    Epoch {epoch}/{EPOCHS}  |  "
              f"Train loss: {tr_loss:.4f}  acc: {tr_acc*100:.2f}%  |  "
              f"Val loss: {vl_loss:.4f}  acc: {vl_acc*100:.2f}%")

    elapsed = time.time() - t0
    results[name] = {
        "train_loss": train_losses, "val_loss": val_losses,
        "train_acc":  train_accs,   "val_acc":  val_accs,
        "time":       elapsed,
        "color":      cfg["color"],
        "category":   cfg["category"],
        "final_val_acc":  val_accs[-1],
        "final_val_loss": val_losses[-1],
    }
    print(f"\n  ✓ {name} done in {elapsed:.1f}s  |  "
          f"Final val acc: {val_accs[-1]*100:.2f}%")


# ──────────────────────────────────────────────────────────────────────────────
# 6.  SUMMARY TABLE
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  RESULTS SUMMARY")
print("=" * 70)
print(f"  {'Optimizer':<22} {'Category':<20} {'Val Acc':>8} {'Val Loss':>10} {'Time(s)':>9}")
print(f"  {'─'*22} {'─'*20} {'─'*8} {'─'*10} {'─'*9}")

sorted_results = sorted(results.items(), key=lambda x: -x[1]["final_val_acc"])
for name, r in sorted_results:
    mark = " ◀ best" if name == sorted_results[0][0] else ""
    print(f"  {name:<22} {r['category']:<20} "
          f"{r['final_val_acc']*100:>7.2f}%  "
          f"{r['final_val_loss']:>10.4f}  "
          f"{r['time']:>8.1f}s{mark}")
print()


# ──────────────────────────────────────────────────────────────────────────────
# 7.  PLOTS
# ──────────────────────────────────────────────────────────────────────────────
epochs_range = list(range(1, EPOCHS + 1))
names  = list(results.keys())
colors = [results[n]["color"] for n in names]

# ── 7a. Training Loss Curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle("Training Loss per Optimizer — MNIST", fontsize=16, fontweight="bold", y=1.01)
axes_flat = axes.flatten()

for i, (name, r) in enumerate(results.items()):
    ax = axes_flat[i]
    ax.plot(epochs_range, r["train_loss"], "o-", color=r["color"],
            linewidth=2, label="Train", markersize=5)
    ax.plot(epochs_range, r["val_loss"], "s--", color=r["color"],
            linewidth=1.5, label="Val", markersize=5, alpha=0.7)
    ax.set_title(name, fontsize=11, fontweight="bold", color=r["color"])
    ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-Entropy Loss")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xticks(epochs_range)

# hide any spare axes
for j in range(len(results), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
path1 = os.path.join(RESULTS_DIR, "01_individual_loss_curves.png")
plt.savefig(path1, dpi=120, bbox_inches="tight")
plt.close()
print(f"  [Plot 1] Saved: {path1}")

# ── 7b. All Val Loss on One Chart ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("All Optimizers Compared — MNIST Validation", fontsize=14, fontweight="bold")

for name, r in results.items():
    ax1.plot(epochs_range, r["val_loss"],  "o-", color=r["color"],
             linewidth=2, label=name, markersize=5)
    ax2.plot(epochs_range, [a*100 for a in r["val_acc"]], "o-", color=r["color"],
             linewidth=2, label=name, markersize=5)

ax1.set_title("Validation Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left"); ax1.grid(True, alpha=0.3)
ax2.set_title("Validation Accuracy (%)"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left"); ax2.grid(True, alpha=0.3)

plt.tight_layout()
path2 = os.path.join(RESULTS_DIR, "02_all_optimizers_comparison.png")
plt.savefig(path2, dpi=120, bbox_inches="tight")
plt.close()
print(f"  [Plot 2] Saved: {path2}")

# ── 7c. Final Bar Chart ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Final Performance Comparison (after all epochs)", fontsize=14, fontweight="bold")

sorted_names  = [n for n, _ in sorted_results]
sorted_colors = [results[n]["color"]           for n in sorted_names]
sorted_accs   = [results[n]["final_val_acc"]*100 for n in sorted_names]
sorted_losses = [results[n]["final_val_loss"]    for n in sorted_names]
sorted_times  = [results[n]["time"]              for n in sorted_names]

bars1 = ax1.barh(sorted_names, sorted_accs, color=sorted_colors, edgecolor="white", linewidth=0.5)
ax1.set_xlabel("Validation Accuracy (%)", fontsize=11)
ax1.set_title("Final Validation Accuracy")
ax1.bar_label(bars1, fmt="%.2f%%", padding=3, fontsize=9)
ax1.set_xlim(min(sorted_accs) - 2, 100)
ax1.grid(True, axis="x", alpha=0.3)

bars2 = ax2.barh(sorted_names, sorted_times, color=sorted_colors, edgecolor="white", linewidth=0.5)
ax2.set_xlabel("Training Time (seconds)", fontsize=11)
ax2.set_title("Training Time")
ax2.bar_label(bars2, fmt="%.1fs", padding=3, fontsize=9)
ax2.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
path3 = os.path.join(RESULTS_DIR, "03_final_bar_comparison.png")
plt.savefig(path3, dpi=120, bbox_inches="tight")
plt.close()
print(f"  [Plot 3] Saved: {path3}")

# ── 7d. Loss Surface Concept — Rosenbrock ─────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Optimizer Trajectories on 2D Rosenbrock Loss Surface\n"
             "f(x,y) = (1-x)² + 100(y-x²)²   |   Minimum at (1,1)",
             fontsize=13, fontweight="bold")

x = np.linspace(-2, 2, 400)
y = np.linspace(-1, 3, 400)
X, Y = np.meshgrid(x, y)
Z = (1 - X)**2 + 100*(Y - X**2)**2
Z_log = np.log(Z + 1)

# Simulate 6 representative optimizer paths
def simulate_path(lr=0.001, beta=0.0, adaptive=False, nadam=False, n=200):
    path = [[-1.5, 0.5]]
    vx, vy = 0.0, 0.0
    gx_sq, gy_sq = 1e-8, 1e-8
    for _ in range(n):
        px, py = path[-1]
        # Rosenbrock gradients
        gx = -2*(1-px) - 400*px*(py - px**2)
        gy =  200*(py - px**2)
        if adaptive:
            gx_sq = 0.99*gx_sq + 0.01*gx**2
            gy_sq = 0.99*gy_sq + 0.01*gy**2
            px_ = px - lr*gx/(np.sqrt(gx_sq)+1e-8)
            py_ = py - lr*gy/(np.sqrt(gy_sq)+1e-8)
        else:
            vx = beta*vx + gx
            vy = beta*vy + gy
            px_ = px - lr*vx
            py_ = py - lr*vy
        px_ = np.clip(px_, -2, 2)
        py_ = np.clip(py_, -1, 3)
        path.append([px_, py_])
    return np.array(path)

optimizer_paths = [
    ("SGD",       simulate_path(lr=0.0005, beta=0.0),           "#e74c3c"),
    ("Momentum",  simulate_path(lr=0.001,  beta=0.9),           "#e67e22"),
    ("Nesterov",  simulate_path(lr=0.001,  beta=0.92),          "#f1c40f"),
    ("RMSprop",   simulate_path(lr=0.005,  adaptive=True),      "#2980b9"),
    ("Adam",      simulate_path(lr=0.05,   adaptive=True),      "#8e44ad"),
    ("AdamW",     simulate_path(lr=0.04,   adaptive=True),      "#6c3483"),
]

for ax, (opt_name, path, col) in zip(axes.flatten(), optimizer_paths):
    contour = ax.contourf(X, Y, Z_log, levels=40, cmap="RdYlBu_r", alpha=0.8)
    ax.contour(X, Y, Z_log, levels=20, colors="white", linewidths=0.3, alpha=0.4)
    ax.plot(path[:, 0], path[:, 1], "-", color=col,
            linewidth=1.5, alpha=0.85, zorder=3)
    ax.plot(path[0, 0], path[0, 1], "o", color="white",
            markersize=7, zorder=5, label="Start")
    ax.plot(path[-1, 0], path[-1, 1], "*", color=col,
            markersize=12, zorder=5, markeredgecolor="white", markeredgewidth=0.8)
    ax.plot(1, 1, "w+", markersize=14, markeredgewidth=2.5, zorder=6, label="Min (1,1)")
    ax.set_title(f"{opt_name}", fontsize=12, fontweight="bold", color=col)
    ax.set_xlim(-2, 2); ax.set_ylim(-1, 3)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(fontsize=7)

plt.tight_layout()
path4 = os.path.join(RESULTS_DIR, "04_loss_surface_trajectories.png")
plt.savefig(path4, dpi=120, bbox_inches="tight")
plt.close()
print(f"  [Plot 4] Saved: {path4}")


# ── 7e. Cheat-Sheet Radar Chart ───────────────────────────────────────────────
categories   = ["Convergence", "Memory\nEfficiency", "Generalization",
                "Sparse\nGradients", "Large Batch\nScaling", "Ease of\nTuning"]
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

ratings = {
    "SGD":            [2, 5, 4, 1, 2, 2],
    "SGD + Momentum": [3, 4, 4, 2, 3, 2],
    "Nesterov":       [4, 4, 4, 2, 3, 2],
    "Adagrad":        [3, 3, 3, 5, 2, 3],
    "Adadelta":       [3, 3, 3, 4, 2, 4],
    "RMSprop":        [4, 3, 3, 4, 3, 3],
    "Adam":           [5, 3, 3, 4, 3, 4],
    "AdamW":          [5, 3, 5, 4, 3, 4],
    "NAdam":          [5, 3, 4, 4, 3, 4],
    "RAdam":          [4, 3, 4, 4, 3, 5],
    "AdaMax":         [4, 3, 3, 5, 3, 4],
}

cols_for_radar = [results[n]["color"] for n in ratings.keys()]
fig = plt.figure(figsize=(20, 16))
fig.suptitle("Optimizer Strengths — Radar Overview", fontsize=16, fontweight="bold", y=1.01)

for idx, (opt_name, vals) in enumerate(ratings.items()):
    ax = fig.add_subplot(3, 4, idx+1, polar=True)
    color = results.get(opt_name, {}).get("color", "#888")
    v = vals + vals[:1]
    ax.plot(angles, v, "o-", linewidth=2, color=color)
    ax.fill(angles, v, alpha=0.25, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=7)
    ax.set_yticks([1, 2, 3, 4, 5]); ax.set_yticklabels(["1","2","3","4","5"], size=6)
    ax.set_ylim(0, 5)
    ax.set_title(opt_name, size=10, fontweight="bold", color=color, pad=12)

# Hide spare subplot
for j in range(len(ratings), 12):
    fig.add_subplot(3, 4, j+1).set_visible(False)

plt.tight_layout()
path5 = os.path.join(RESULTS_DIR, "05_radar_cheat_sheet.png")
plt.savefig(path5, dpi=120, bbox_inches="tight")
plt.close()
print(f"  [Plot 5] Saved: {path5}")


# ──────────────────────────────────────────────────────────────────────────────
# 8.  DECISION GUIDE
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  WHICH OPTIMIZER SHOULD I USE? — Decision Guide")
print("=" * 70)

guide = """
  ┌──────────────────────────────────────────────────────────────────────┐
  │  TASK / SITUATION                 │  RECOMMENDED OPTIMIZER          │
  ├──────────────────────────────────────────────────────────────────────┤
  │  Transformers / LLMs / BERT/GPT   │  AdamW  (decoupled decay)       │
  │  CNNs (vision, final accuracy)    │  SGD + Momentum + LR schedule   │
  │  RNNs / LSTMs / time-series       │  RMSprop or Adam                │
  │  NLP with sparse embeddings       │  Adagrad or AdaMax              │
  │  General / unsure where to start  │  Adam  (reliable default)       │
  │  Large batch training (BERT-like) │  LAMB  (layer-wise trust ratio) │
  │  No warmup scheduler desired      │  RAdam  (auto-warmup)           │
  │  Slight edge over Adam needed     │  NAdam  (Nesterov + Adam)       │
  │  Best final generalisation        │  SGD + Momentum + CosineAnnealing│
  │  Sparse data, online learning     │  Adadelta  (no manual LR)       │
  └──────────────────────────────────────────────────────────────────────┘

  GENERAL TIPS
  ────────────
  1. Start with Adam (lr=1e-3). Switch to AdamW when using weight decay.
  2. If final accuracy matters more than speed, re-train with SGD+Momentum.
  3. Use a learning rate scheduler with all optimizers:
       • StepLR / MultiStepLR  → simple, manual drops
       • CosineAnnealingLR     → smooth decay, best for SGD
       • OneCycleLR            → fast convergence, works with Adam too
  4. Gradient clipping (clip_grad_norm_) is recommended with Adam for RNNs.
  5. AdamW + warmup + cosine decay is the standard recipe for Transformers.
"""
print(guide)

print("=" * 70)
print(f"  All plots saved to: ./{RESULTS_DIR}/")
print("  Files:")
for p in [path1, path2, path3, path4, path5]:
    print(f"    • {p}")
print("=" * 70)
print("  Done! 🎉")
print("=" * 70)

  DEEP LEARNING OPTIMIZERS — Comprehensive Guide
  Device  : cuda
  Epochs  : 5   |   Batch size: 256
  Dataset : MNIST (60 000 train / 10 000 test)

[1] Loading MNIST dataset …


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 467kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.63MB/s]


    Train samples : 60,000
    Test  samples : 10,000
    Input shape   : 28×28 grayscale → flattened to 784

[2] Model: 3-layer MLP  |  Trainable params: 235,914

  TRAINING — running each optimizer for 5 epochs

──────────────────────────────────────────────────────────────────────
STOCHASTIC GRADIENT DESCENT (SGD)
  ──────────────────────────────────
  The simplest optimizer. Computes gradient of loss w.r.t. each parameter
  and takes a small step in the negative gradient direction.

  Update rule:   θ ← θ − η · ∇L(θ; x_batch)

  Key hyperparameters:
    η  (lr)          : step size (typical: 0.01 – 0.1)

  Intuition: Imagine standing on a hilly landscape (loss surface) and always
  stepping downhill. With full-batch GD every step uses ALL data (expensive).
  SGD uses a random mini-batch, introducing noise that can actually help
  escape local minima — but convergence is slower and oscillatory.

  Best for: Final fine-tuning, when combined with a good LR schedule.
  Caution : Very s

In [ ]:
Optimizer Techniques

1. SGD
2. SGD + Momentum
3. Nesterov
4. Adagrad
5. Adadelta
6. RMSProp
7. Adam
8. AdamW
9. NAdam
10. RAdam
11. AdaMax